In [1]:
import scanpy as sc
import anndata as ad
import numpy as np
from sklearn.utils import shuffle
from lightgbm import LGBMClassifier
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score
import shap
import joblib

/home/michal_sieczczynski/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
adata_train_healthy = sc.read_h5ad("./ready_data/train_healthy_fixed.h5ad")
adata_train_tumor = sc.read_h5ad("./ready_data/train_tumor_fixed.h5ad")

adata_train_healthy.obs['label'] = 'PBMC'
adata_train_tumor.obs['label'] = 'Tumor'

adata_train = ad.concat([adata_train_healthy, adata_train_tumor], join='inner')

X_train = adata_train.X
y_train = adata_train.obs['label'].values

X_train, y_train = shuffle(X_train, y_train, random_state=42)

In [8]:
adata_train.obs['label'].value_counts()

label
PBMC     546635
Tumor     42420
Name: count, dtype: int64

In [3]:
adata_test_healthy = sc.read_h5ad("./ready_data/test_healthy_fixed.h5ad")
adata_test_tumor = sc.read_h5ad("./ready_data/test_tumor_fixed.h5ad")

adata_test_healthy.obs['label'] = 'PBMC'
adata_test_tumor.obs['label'] = 'Tumor'

adata_test = ad.concat([adata_test_healthy, adata_test_tumor], join='inner')

X_test = adata_test.X
y_test = adata_test.obs['label'].values

In [9]:
adata_test.obs['label'].value_counts()

label
PBMC     33783
Tumor    24160
Name: count, dtype: int64

In [4]:
adata_val_healthy = sc.read_h5ad("./ready_data/val_healthy_fixed.h5ad")
adata_val_tumor = sc.read_h5ad("./ready_data/val_tumor_fixed.h5ad")

adata_val_healthy.obs['label'] = 'PBMC'
adata_val_tumor.obs['label'] = 'Tumor'

adata_val = ad.concat([adata_val_healthy, adata_val_tumor], join='inner')

X_val = adata_val.X
y_val = adata_val.obs['label'].values

In [10]:
adata_val.obs['label'].value_counts()

label
PBMC     60488
Tumor    24773
Name: count, dtype: int64

In [5]:
adata_test_ctc = sc.read_h5ad("./ready_data/ctc_normalized_data_fixed.h5ad")
adata_test_ctc.obs['label'] = 'Tumor'

X_test_ctc = adata_test_ctc.X
y_test_ctc = adata_test_ctc.obs['label'].values

In [11]:
adata_test_ctc.obs['label'].value_counts()

label
Tumor    1038
Name: count, dtype: int64

In [12]:
lightgbm_object = LGBMClassifier(num_leaves=22, max_depth=13, is_unbalance=False,
                                learning_rate=0.031109, n_estimators=271, random_state=42, verbosity=-1,
                                reg_alpha=0.0, reg_lambda=0.0, min_child_samples=45)
# params_learning_rate                                       0.031109
# params_max_depth                                                 13
# params_min_child_samples                                         45
# params_n_estimators                                             271
# params_num_leaves                                                22
# params_reg_alpha                                                0.0
# params_reg_lambda                                               0.0
lightgbm_object.fit(X_train, y_train)

LGBMClassifier(is_unbalance=False, learning_rate=0.031109, max_depth=13,
               min_child_samples=45, n_estimators=271, num_leaves=22,
               random_state=42, verbosity=-1)

In [14]:
gene_names = adata_test_ctc.var_names.to_numpy()

importance_df = pd.DataFrame({
    "gene": gene_names,
    "importance": lightgbm_object.feature_importances_
})

importance_df = importance_df[
    importance_df["importance"] > 0
].sort_values(
    "importance",
    ascending=False
)
print(importance_df)

           gene  importance
3327    EPB41L1         282
8995      RPL13         270
9763    SLC33A1         241
3127      EDNRB         240
6576      MYO1B         238
...         ...         ...
5369      KIF5B           1
5459      KPNA6           1
5523    LAMTOR3           1
5634      LIMS1           1
4669  HNRNPA2B1           1

[238 rows x 2 columns]


In [7]:
joblib.dump(lightgbm_object, "lightgbm_final_model_fixed.pkl")

['lightgbm_final_model_fixed.pkl']

In [8]:
y_pred = lightgbm_object.predict(X_val)
print(f"Balanced accuracy: {balanced_accuracy_score(y_val, y_pred)}")
print(f"Precision for Tumor: {precision_score(y_val, y_pred, pos_label='Tumor')}")
print(f"Precision for PBMC: {precision_score(y_val, y_pred, pos_label='PBMC')}")
print(f"ROC AUC: {roc_auc_score(y_val, lightgbm_object.predict_proba(X_val)[:,1])}")

cm = confusion_matrix(y_val, y_pred)
print(cm)

Balanced accuracy: 1.0
Precision for Tumor: 1.0
Precision for PBMC: 1.0
ROC AUC: 1.0
[[60488     0]
 [    0 24773]]


In [9]:
y_pred = lightgbm_object.predict(X_test)
print(f"Balanced accuracy: {balanced_accuracy_score(y_test, y_pred)}")
print(f"Precision for Tumor: {precision_score(y_test, y_pred, pos_label='Tumor')}")
print(f"Precision for PBMC: {precision_score(y_test, y_pred, pos_label='PBMC')}")
print(f"ROC AUC: {roc_auc_score(y_test, lightgbm_object.predict_proba(X_test)[:,1])}")

cm = confusion_matrix(y_test, y_pred)
print(cm)

Balanced accuracy: 0.9999793046357616
Precision for Tumor: 1.0
Precision for PBMC: 0.9999704001894388
ROC AUC: 1.0
[[33783     0]
 [    1 24159]]


In [10]:
y_pred = lightgbm_object.predict(X_test_ctc)
print(f"Balanced accuracy: {balanced_accuracy_score(y_test_ctc, y_pred)}")
print(f"Recall: {recall_score(y_test_ctc, y_pred, pos_label='Tumor')}")

cm = confusion_matrix(y_test_ctc, y_pred)
print(cm)

Balanced accuracy: 0.9980732177263969
Recall: 0.9980732177263969
[[   0    0]
 [   2 1036]]


/home/michal_sieczczynski/miniconda3/envs/sequencing/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2480: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [11]:
gene_names = adata_test_ctc.var_names.tolist()
explainer = shap.TreeExplainer(lightgbm_object)
shap_values = explainer(X_test).values

global_shap = np.abs(shap_values).mean(axis=0).A1
shap_df = pd.DataFrame({
    "gene": gene_names,
    "SHAP": global_shap
})

shap_df = shap_df[shap_df["SHAP"] != 0]
shap_df = shap_df.sort_values("SHAP", ascending=False)

shap_df.to_csv("feature_importance/lightgbm_shap_values_tumor_test_fixed.csv", index=False)

In [12]:
shap_values = explainer(X_test_ctc).values

global_shap = np.abs(shap_values).mean(axis=0).A1
shap_df = pd.DataFrame({
    "gene": gene_names,
    "SHAP": global_shap
})

shap_df = shap_df[shap_df["SHAP"] != 0]
shap_df = shap_df.sort_values("SHAP", ascending=False)

shap_df.to_csv("feature_importance/lightgbm_shap_values_ctc_test_fixed.csv", index=False)